# Lab 4.4 &mdash; MCP From the Wire Up

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 40 min &nbsp;|&nbsp; **Day 2 &middot; Module 4 &mdash; Tool Calling &amp; MCP**

### What you'll do
- Publish your own <code>@tool</code> objects as MCP tools, using the SDK's own types
- Frame a JSON-RPC message the way MCP does, and find out why framing exists at all
- Write the server: initialize, tools/list, tools/call &mdash; and where failures belong
- Read an <code>mcpServers</code> config as what it is: a list of access grants

> **How this lab works.** You write real LangChain and MCP code. Fill every `BLANK`, then run
> the **Self-check** cell under each section &mdash; those assert on the *objects you built*
> (a `@tool`, an argument schema, a `ToolMessage`, an `mcp.types.Tool`), so they are
> deterministic and do not depend on the model. Cells marked **Run it for real** put your code
> in front of the sandbox model; that is the part worth watching. The score line is feedback,
> not a grade.

> **You implement the protocol.** The message *types* come from the `mcp` package, so
> the SDK validates every response you build; the transport you write yourself. The
> last cell runs a server as a real subprocess &mdash; no model, no network.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-4-04")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nSelf-check: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model can reason before it answers, and that reasoning is billed as completion
# tokens. It is off here because tool selection is a short decision and you will make a lot
# of them today. Pass think=True to see the difference for yourself.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

def show_messages(messages, width: int = 88) -> None:
    """Print a message list the way a trace reads: type, content, and any tool calls."""
    for m in messages:
        kind = getattr(m, "type", "?")
        body = str(getattr(m, "content", "")).replace("\n", " ")[:width]
        calls = getattr(m, "tool_calls", None)
        line = f"  [{kind:9}] {body}"
        if calls:
            line += "  -> calls: " + ", ".join(f"{c['name']}({c['args']})" for c in calls)
        print(line)

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

In [ ]:
# ------------------------------------------------- the case file (synthetic, self-contained)
# One domain runs through all five Module 4 labs -- the same payment exceptions as Day 1,
# now reached through tools the model chooses, and then through tools you did not write.
# Nothing here is real data and nothing leaves this notebook.

LEDGER = {
    "PMT-1001": {"amount": 250000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "settled",  "value_date": "2026-09-01", "reason_code": None},
    "PMT-1002": {"amount":  48250.75, "ccy": "EUR", "counterparty": "ACME-EU",
                 "status": "failed",   "value_date": "2026-09-02", "reason_code": "INSUFFICIENT_FUNDS"},
    "PMT-1003": {"amount": 990000.00, "ccy": "USD", "counterparty": "ZENITH",
                 "status": "held",     "value_date": "2026-09-02", "reason_code": "LIMIT_BREACH"},
    "PMT-1004": {"amount":   1200.00, "ccy": "GBP", "counterparty": "ACME-UK",
                 "status": "failed",   "value_date": "2026-09-03", "reason_code": "INVALID_IBAN"},
    "PMT-1005": {"amount": 750000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "held",     "value_date": "2026-09-03", "reason_code": "SANCTIONS_REVIEW"},
}

POLICY = {
    "INSUFFICIENT_FUNDS": "Retry once after 24h. If it fails again, notify the client desk. No manual funding.",
    "LIMIT_BREACH":       "Payments above USD 500,000 need Treasury approval before release.",
    "INVALID_IBAN":       "Return to originator with code R04. Never repair beneficiary details in-house.",
    "SANCTIONS_REVIEW":   "Hold. Compliance decides. Operations must not release or cancel.",
}

# Which reason codes may an agent resolve on its own, and which need a human?
NEEDS_HUMAN = {"LIMIT_BREACH", "SANCTIONS_REVIEW"}

print(f"{len(LEDGER)} payments, {len(POLICY)} policy rules loaded")

In [ ]:
# ------------------------------------------------- the toolkit (nothing to fill in)
# Four tools over that ledger, written with LangChain's @tool decorator. Three read; one
# moves money -- the distinction that starts mattering the moment a model is choosing.
# Read the docstrings properly: they are not comments, they are the API the model sees.
from langchain_core.tools import tool

@tool
def lookup_payment(ref: str) -> str:
    """Return the ledger record for one payment reference such as 'PMT-1002'.

    Use when you already have the reference. Not for searching across payments --
    use search_payments when you do not have one.
    """
    record = LEDGER.get(ref)
    if record is None:
        return f"no payment found with reference {ref!r}"
    return json.dumps({"ref": ref, **record})


@tool
def search_payments(counterparty: str = "", status: str = "") -> str:
    """Return every ledger record matching a counterparty, a status, or both.

    Use when you must find which payments match. Not for one known reference --
    use lookup_payment for that.
    """
    hits = [{"ref": r, **v} for r, v in LEDGER.items()
            if (not counterparty or v["counterparty"] == counterparty)
            and (not status or v["status"] == status)]
    return json.dumps(hits)


@tool
def policy_for(reason_code: str) -> str:
    """Return the operating policy for one failure reason code such as 'LIMIT_BREACH'.

    Use once you know why a payment failed and need to know what to do about it.
    """
    return POLICY.get(reason_code, f"no policy on file for reason code {reason_code!r}")


@tool
def release_payment(ref: str) -> str:
    """Release one held payment so that it settles. This one moves money.

    Use only after a named human has approved this specific release. Not for reading,
    searching or explaining.
    """
    record = LEDGER.get(ref)
    if record is None:
        return f"no payment found with reference {ref!r}"
    return json.dumps({"ref": ref, "released": True, "was": record["status"]})


TOOLKIT = [lookup_payment, search_payments, policy_for, release_payment]
BY_NAME = {t.name: t for t in TOOLKIT}
print("toolkit:", ", ".join(BY_NAME))

## Concept

MCP is JSON-RPC 2.0 in both directions over a transport. Over stdio there is no HTTP to tell the
reader where one message ends, so each is **framed** with a `Content-Length` header &mdash; the same
trick the Language Server Protocol uses, for the same reason.

Three methods carry almost everything:

| method | what it does |
|---|---|
| `initialize` | agree a protocol version and exchange capabilities |
| `tools/list` | **discovery** &mdash; the client learns the tools at run time |
| `tools/call` | invoke one by name with arguments |

Discovery is the part with consequences. The agent does not know what it can do until it asks,
which is what lets a server gain a tool without your redeploying &mdash; and what makes a server
you did not review a problem you did not review.

## Section 1 &mdash; The protocol is a schema you can import

Nothing about MCP has to be reverse-engineered. The `mcp` package ships every message as a
Pydantic model, so a malformed response fails where you built it rather than at the far end.

Look at what an MCP `Tool` needs: a name, a description, and a JSON Schema for the arguments.
That is Lab 4.1's three fields, over a wire.

In [ ]:
from mcp.types import (Tool, TextContent, CallToolResult, ListToolsResult,
                       InitializeResult, Implementation, ServerCapabilities,
                       LATEST_PROTOCOL_VERSION)

def input_schema(t) -> dict:
    """The JSON Schema for one LangChain tool's arguments."""
    schema = t.args_schema
    return schema if isinstance(schema, dict) else schema.model_json_schema()


def as_mcp_tool(t) -> Tool:
    """Publish one of your LangChain tools the way MCP describes it."""
    return Tool(
        name=t.name,
        # TODO: an MCP client reads this to decide whether to call the tool, exactly as a
        #       bound model does. Which field of your tool carries that text?
        description=BLANK,
        inputSchema=input_schema(t),
    )

In [ ]:
# --- Self-check: Section 1   (MCP model objects only -- no server, no model call)
check("the result is a real MCP Tool, validated by the SDK's own schema",
      lambda: isinstance(as_mcp_tool(lookup_payment), Tool))
check("the name crosses unchanged",
      lambda: as_mcp_tool(lookup_payment).name == "lookup_payment")
check("your description crosses whole, boundary sentence and all",
      lambda: "Not for searching" in as_mcp_tool(lookup_payment).description,
      "over MCP that sentence is the only thing standing between two tools that read alike")
check("the argument schema crosses too, required arguments and all",
      lambda: as_mcp_tool(lookup_payment).inputSchema["required"] == ["ref"])
check("an optional argument is not marked required",
      lambda: "counterparty" not in
              (as_mcp_tool(search_payments).inputSchema.get("required") or []))
check("a whole toolkit is a ListToolsResult",
      lambda: len(ListToolsResult(tools=[as_mcp_tool(t) for t in TOOLKIT]).tools) == 4)
check("and it serialises to the JSON that goes on the wire",
      lambda: "inputSchema" in json.dumps(
          as_mcp_tool(lookup_payment).model_dump(mode="json", by_alias=True, exclude_none=True)))

guard(lambda: print(json.dumps(
    as_mcp_tool(policy_for).model_dump(mode="json", by_alias=True, exclude_none=True),
    indent=2)[:460]))

## Section 2 &mdash; Framing

Nothing to fill in here &mdash; read it instead, because the one detail that matters is easy to
miss. The header counts **bytes**, not characters. One non-ASCII character and the two differ,
after which every following message in the stream is read from the wrong offset.

In [ ]:
import re

def encode(message: dict) -> bytes:
    """Frame one JSON-RPC message for the stdio transport."""
    body = json.dumps(message, ensure_ascii=False).encode("utf-8")
    return f"Content-Length: {len(body)}\r\n\r\n".encode("ascii") + body


def decode_all(blob: bytes) -> list:
    """Every complete message in a byte stream -- which is what framing makes possible."""
    out, i = [], 0
    while True:
        j = blob.find(b"\r\n\r\n", i)
        if j < 0:
            return out
        n = int(re.search(r"Content-Length:\s*(\d+)", blob[i:j].decode("ascii")).group(1))
        start = j + 4
        out.append(json.loads(blob[start:start + n]))
        i = start + n

In [ ]:
# --- Self-check: Section 2   (bytes only)
_m = {"jsonrpc": "2.0", "id": 1, "method": "tools/list", "params": {}}
_uni = {"jsonrpc": "2.0", "id": 2, "method": "tools/call",
        "params": {"arguments": {"counterparty": "CAFÉ-EU"}}}

check("a message survives a round trip", lambda: decode_all(encode(_m)) == [_m])
check("the header names Content-Length",
      lambda: encode(_m).split(b"\r\n")[0].startswith(b"Content-Length:"))
check("two messages in one stream decode as two",
      lambda: decode_all(encode(_m) + encode(_m)) == [_m, _m])
check("the length counts BYTES, not characters",
      lambda: int(re.search(rb"Content-Length: (\d+)", encode(_uni)).group(1))
              > len(json.dumps(_uni, ensure_ascii=False)),
      "E-acute is one character and two bytes -- count characters and every later message "
      "in the stream is read from the wrong offset")
check("and a non-ASCII payload still round-trips inside a stream",
      lambda: decode_all(encode(_uni) + encode(_m)) == [_uni, _m])

## Section 3 &mdash; The server

Note where tool failures go. A tool that could not do its job is a **successful** JSON-RPC
response carrying `isError: true` &mdash; because the protocol worked perfectly. A JSON-RPC
`error` means the *protocol* failed: unknown method, malformed request.

Collapsing the two is the most common MCP implementation bug, and it makes tool failures
invisible to the model: the client sees a transport error, drops the content, and the model never
learns that the payment does not exist.

In [ ]:
SERVER_TOOLS = {"lookup_payment": lookup_payment, "policy_for": policy_for}

def _result(rid, payload) -> dict:
    """One successful JSON-RPC response. `payload` is an MCP result model."""
    return {"jsonrpc": "2.0", "id": rid,
            "result": payload.model_dump(mode="json", by_alias=True, exclude_none=True)}


def _rpc_error(rid, code, message) -> dict:
    """A PROTOCOL failure. Nothing a tool does belongs in here."""
    return {"jsonrpc": "2.0", "id": rid, "error": {"code": code, "message": message}}


def call_tool(name: str, arguments: dict) -> CallToolResult:
    """Run one tool and answer in MCP's shape."""
    t = SERVER_TOOLS.get(name)
    if t is None:
        return CallToolResult(content=[TextContent(type="text", text=f"no such tool: {name!r}")],
                              isError=True)
    try:
        text, failed = str(t.invoke(arguments)), False
    except Exception as exc:
        text, failed = f"{type(exc).__name__}: {exc}", True

    # TODO: the protocol worked; only the tool may not have. Which of the two names above
    #       says so? (Get this wrong and every failure reads as a success.)
    return CallToolResult(content=[TextContent(type="text", text=text)], isError=BLANK)


def handle(request: dict) -> dict:
    """One JSON-RPC request in, one response out. This is the entire server."""
    rid, method = request.get("id"), request.get("method")
    params = request.get("params") or {}

    if method == "initialize":
        return _result(rid, InitializeResult(
            protocolVersion=LATEST_PROTOCOL_VERSION,
            capabilities=ServerCapabilities(),
            serverInfo=Implementation(name="ledger", version="1.0.0")))

    if method == "tools/list":
        return _result(rid, ListToolsResult(
            tools=[as_mcp_tool(t) for t in SERVER_TOOLS.values()]))

    if method == "tools/call":
        return _result(rid, call_tool(params.get("name"), params.get("arguments") or {}))

    return _rpc_error(rid, -32601, f"method not found: {method}")

In [ ]:
# --- Self-check: Section 3   (your server, in process -- no model call)
def _req(method, **params) -> dict:
    return {"jsonrpc": "2.0", "id": 7, "method": method, "params": params}

def _call(**args) -> dict:
    return handle(_req("tools/call", **args))["result"]

check("initialize agrees the protocol version the SDK ships with",
      lambda: handle(_req("initialize"))["result"]["protocolVersion"] == LATEST_PROTOCOL_VERSION)
check("and names the server",
      lambda: handle(_req("initialize"))["result"]["serverInfo"]["name"] == "ledger")
check("tools/list publishes name, description and inputSchema for every tool",
      lambda: all({"name", "description", "inputSchema"} <= set(t)
                  for t in handle(_req("tools/list"))["result"]["tools"]))
check("the descriptions on the wire are your real ones",
      lambda: "Not for searching" in json.dumps(handle(_req("tools/list"))["result"]))
check("a good call returns the record as text content",
      lambda: "INSUFFICIENT_FUNDS" in
              _call(name="lookup_payment", arguments={"ref": "PMT-1002"})["content"][0]["text"])
check("a good call is not flagged as an error",
      lambda: _call(name="lookup_payment", arguments={"ref": "PMT-1002"})["isError"] is False)
check("an unknown tool is a RESULT with isError, not a JSON-RPC error",
      lambda: _call(name="nope", arguments={})["isError"] is True,
      "the protocol worked -- only the tool did not; collapsing these hides failures from the model")
check("a tool that raises is caught and reported as isError",
      lambda: _call(name="lookup_payment", arguments={"wrong_arg": 1})["isError"] is True)
check("nothing escapes the server as an exception",
      lambda: isinstance(handle(_req("tools/call", name="lookup_payment", arguments={})), dict))
check("an unknown METHOD is a real JSON-RPC error",
      lambda: handle(_req("tools/nonesuch"))["error"]["code"] == -32601,
      "this one really is a protocol failure, so it belongs in the error channel")
check("every response you build validates against the SDK's own model",
      lambda: CallToolResult.model_validate(
          _call(name="lookup_payment", arguments={"ref": "PMT-1003"})).isError is False)

## Section 4 &mdash; The client, and discovery

The client below sends every message through `encode`/`decode_all`, so it is talking over the real
wire format even while the server is in the same process. Swapping in a pipe changes nothing above
the transport &mdash; which the last cell proves.

Nothing to fill in. Watch what `list_tools` does: the client did not know a single tool name
until that call returned.

In [ ]:
class Session:
    """An MCP client session against one server."""

    def __init__(self, handler):
        self._handler, self._id, self.tools = handler, 0, []

    def request(self, method: str, params: dict = None) -> dict:
        self._id += 1
        message = {"jsonrpc": "2.0", "id": self._id, "method": method, "params": params or {}}
        [on_the_wire] = decode_all(encode(message))       # framed and parsed, as over a pipe
        return self._handler(on_the_wire)

    def initialize(self) -> InitializeResult:
        return InitializeResult.model_validate(self.request("initialize")["result"])

    def list_tools(self) -> list:
        """Discovery. The client did not know these names until this call returned."""
        payload = self.request("tools/list")["result"]
        self.tools = ListToolsResult.model_validate(payload).tools
        return self.tools

    def call_tool(self, name: str, **arguments) -> dict:
        payload = self.request("tools/call", {"name": name, "arguments": arguments})["result"]
        result = CallToolResult.model_validate(payload)
        return {"text": result.content[0].text, "is_error": bool(result.isError)}

In [ ]:
# --- Self-check: Section 4   (client and server, in process -- no model call)
def _session() -> Session:
    s = Session(handle)
    s.initialize()
    s.list_tools()
    return s

check("the session knows nothing about the tools before it asks",
      lambda: Session(handle).tools == [],
      "discovery at run time is what lets a server change without your redeploying")
check("and knows both of them afterwards",
      lambda: {t.name for t in _session().tools} == {"lookup_payment", "policy_for"})
check("what came back are MCP Tool objects, not loose dicts",
      lambda: all(isinstance(t, Tool) for t in _session().tools))
check("each request carries a fresh id",
      lambda: _session()._id == 2)
check("a tool call returns the text",
      lambda: "ZENITH" in _session().call_tool("lookup_payment", ref="PMT-1003")["text"])
check("and is not flagged as an error",
      lambda: _session().call_tool("lookup_payment", ref="PMT-1003")["is_error"] is False)
check("a failed call surfaces as is_error rather than an exception",
      lambda: _session().call_tool("nope")["is_error"] is True)
check("the second tool works through the same session",
      lambda: "Treasury approval" in
              _session().call_tool("policy_for", reason_code="LIMIT_BREACH")["text"])

def _show_discovery():
    for t in _session().tools:
        print(f"  {t.name:16} {t.description.splitlines()[0][:62]}")
guard(_show_discovery)

## Section 5 &mdash; The config is the grant

Four lines of JSON give an agent a capability. Nothing in the agent's code changes, nothing is
compiled, and by default nothing reviews it. So read the file the way you would read an IAM
policy: **which of these entries lets the agent change something?**

In [ ]:
CONFIG = {
    "mcpServers": {
        "ledger":  {"command": "python", "args": ["-m", "ledger_mcp"],
                    "env": {"LEDGER_SCOPE": "read-only"}},
        "policy":  {"command": "python", "args": ["-m", "policy_mcp"],
                    "env": {"POLICY_SCOPE": "read-only"}},
        "release": {"command": "python", "args": ["-m", "release_mcp"],
                    "env": {"RELEASE_SCOPE": "write"}},
        "notes":   {"command": "python", "args": ["-m", "notes_mcp"]},
    }
}

def write_scopes() -> set:
    """The scope values that mean a server can CHANGE something."""
    # TODO: of the values that turn up in configs like these -- "read-only", "write",
    #       "read-write", "admin" -- which ones grant the power to change something?
    return BLANK


def servers_that_can_write(config: dict) -> list:
    """The configured servers that grant the agent that power."""
    out = []
    for name, entry in config["mcpServers"].items():
        scopes = {str(v).lower() for v in (entry.get("env") or {}).values()}
        if scopes & write_scopes():
            out.append(name)
    return sorted(out)

In [ ]:
# --- Self-check: Section 5   (config only)
_with_admin = {"mcpServers": {**CONFIG["mcpServers"],
                              "ops": {"command": "python", "args": ["-m", "ops_mcp"],
                                      "env": {"OPS_SCOPE": "admin"}}}}

check("exactly one configured server can write today",
      lambda: servers_that_can_write(CONFIG) == ["release"])
check("read-only is not a write grant",
      lambda: "ledger" not in servers_that_can_write(CONFIG))
check("an admin scope is a write grant too",
      lambda: servers_that_can_write(_with_admin) == ["ops", "release"])
check("a server with no env declared is not treated as a write grant",
      lambda: "notes" not in servers_that_can_write(CONFIG),
      "it is also the one you know least about -- undeclared is not the same as safe")
check("the scope lives in the config, not in the agent's code",
      lambda: all("SCOPE" in k
                  for e in CONFIG["mcpServers"].values() for k in (e.get("env") or {})),
      "which is what makes it reviewable and revocable without touching the agent")

def _grants():
    for name, entry in CONFIG["mcpServers"].items():
        env = entry.get("env") or {}
        print(f"  {name:9} {' '.join([entry['command']] + entry['args']):24} "
              f"{'WRITE' if name in servers_that_can_write(CONFIG) else 'read':>6}  {env}")
guard(_grants)

### The server you are about to launch

Small enough to read in a minute, which is the point. Same three methods, same framing, stdlib
only, and its own private copy of a ledger &mdash; it shares nothing with this notebook.

In [ ]:
MCP_SERVER_SOURCE = r"""
import sys, json, re

LEDGER = {
    "PMT-1002": {"amount": 48250.75, "ccy": "EUR", "counterparty": "ACME-EU",
                 "status": "failed", "reason_code": "INSUFFICIENT_FUNDS"},
    "PMT-1003": {"amount": 990000.00, "ccy": "USD", "counterparty": "ZENITH",
                 "status": "held", "reason_code": "LIMIT_BREACH"},
}
SPECS = [{"name": "lookup_payment",
          "description": "Return the ledger record for one payment reference such as PMT-1002.",
          "inputSchema": {"type": "object", "properties": {"ref": {"type": "string"}},
                          "required": ["ref"]}}]

def framed(msg):
    body = json.dumps(msg).encode("utf-8")
    return b"Content-Length: " + str(len(body)).encode() + b"\r\n\r\n" + body

def handle(req):
    rid, method = req.get("id"), req.get("method")
    params = req.get("params") or {}
    if method == "initialize":
        return {"jsonrpc": "2.0", "id": rid,
                "result": {"protocolVersion": "2025-06-18", "capabilities": {"tools": {}},
                           "serverInfo": {"name": "ledger", "version": "1.0.0"}}}
    if method == "tools/list":
        return {"jsonrpc": "2.0", "id": rid, "result": {"tools": SPECS}}
    if method == "tools/call":
        ref = (params.get("arguments") or {}).get("ref")
        rec = LEDGER.get(ref)
        text = json.dumps({"ref": ref, **rec}) if rec else "no payment found with reference %r" % ref
        return {"jsonrpc": "2.0", "id": rid,
                "result": {"content": [{"type": "text", "text": text}], "isError": rec is None}}
    return {"jsonrpc": "2.0", "id": rid,
            "error": {"code": -32601, "message": "method not found"}}

data, out, i = sys.stdin.buffer.read(), b"", 0
while True:
    j = data.find(b"\r\n\r\n", i)
    if j < 0:
        break
    n = int(re.search(rb"Content-Length:\s*(\d+)", data[i:j]).group(1))
    start = j + 4
    out += framed(handle(json.loads(data[start:start + n])))
    i = start + n
sys.stdout.buffer.write(out)
"""
print(f"{len(MCP_SERVER_SOURCE.splitlines())} lines of server")

## Run it for real &mdash; over a real pipe

No model and no network needed for this one. The cell writes that server to your work directory,
launches it as a **separate process**, and talks to it over stdin and stdout with the framing from
Section 2.

Everything above the transport is the same code. That is the claim the protocol makes, and this
is it being true.

In [ ]:
def talk_to_a_real_server():
    import subprocess, sys as _sys
    path = os.path.join(WORK, "ledger_mcp_server.py")
    with open(path, "w") as fh:
        fh.write(MCP_SERVER_SOURCE)

    payload = b"".join(encode(m) for m in [
        {"jsonrpc": "2.0", "id": 1, "method": "initialize", "params": {}},
        {"jsonrpc": "2.0", "id": 2, "method": "tools/list", "params": {}},
        {"jsonrpc": "2.0", "id": 3, "method": "tools/call",
         "params": {"name": "lookup_payment", "arguments": {"ref": "PMT-1003"}}},
    ])
    proc = subprocess.run([_sys.executable, path], input=payload,
                          capture_output=True, timeout=60)
    if proc.returncode != 0:
        print("server exited", proc.returncode, proc.stderr.decode()[:300])
        return

    for msg in decode_all(proc.stdout):
        result = msg.get("result", {})
        if "serverInfo" in result:
            print(f"  initialize -> {result['serverInfo']} protocol {result['protocolVersion']}")
        elif "tools" in result:
            print(f"  tools/list -> discovered {[t['name'] for t in result['tools']]}")
        elif "content" in result:
            print(f"  tools/call -> {result['content'][0]['text'][:88]}")

guard(talk_to_a_real_server)

### Read it

That was a real process boundary: a separate interpreter, its own memory, its own environment, and
nothing shared with this notebook but two pipes. Give it different credentials and you have the
governance story from the deck &mdash; a tool you can grant, revoke and audit on its own.

You read that server before you ran it. Ask yourself what you actually know about a server you
install from a registry with one line of JSON &mdash; and carry the question into Lab 4.5.

In [ ]:
score()

## Your turn

1. Point `SERVER_TOOLS` at all four tools instead of two and re-run Section 4. You just granted
   an agent the ability to release payments, and the diff was one line in a dict.
2. Add `resources/list` and `resources/read`, and move `policy_for` behind a resource instead of a
   tool. Which agent behaviours become impossible &mdash; and is that a loss or the point?
3. Make the subprocess server emit a `Content-Length` ten bytes too long, and watch `decode_all`
   quietly return fewer messages than you sent. Where does the timeout belong, and what should a
   client do about a truncated stream?